## Imports

In [ ]:
from keras import layers, optimizers, Input, Model, callbacks, regularizers
from keras.models import load_model
import keras
from methods.load_data import load_ucsf
from methods.visualisation import plot_training_history, plot_augmentation

IMAGE_SIZE = 128

## Get dataset

In [ ]:
x_train, y_train, x_val, y_val, x_test, y_test = load_ucsf(debug=False, classification=True, image_size=IMAGE_SIZE)
print(f"Training shape: x_train - {x_train.shape}, y_train - {y_train.shape}")
print(f"Validation shape: x_val - {x_val.shape}, y_val - {y_val.shape}")
print(f"Testing shape: x_test - {x_test.shape}, y_test - {y_test.shape}")

## Add Augmentation

In [ ]:
augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.03),                
    layers.RandomTranslation(0.05, 0.05),        
    layers.RandomZoom(0.1),
], name="augmentation")

plot_augmentation(x_train, 4, augmentation)

## Load model

In [ ]:
base_model = load_model("models/best_classification_model.keras")
inputs = Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
outputs = base_model(augmentation(inputs))
finetune_model = Model(inputs, outputs)

## Compile model

In [ ]:
finetune_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.00001),
    loss="binary_crossentropy",
    metrics=["AUC"]
)

## Baseline model AUC

In [ ]:
_, baseline_val_auc = finetune_model.evaluate(x_val, y_val, verbose=0)
print(f"Validation AUC before fine tuning: {baseline_val_auc:.4f}")

## Fit model

In [ ]:
BATCH_SIZE = 32
EPOCHS = 20

def get_callbacks():
    return [
        callbacks.EarlyStopping(
            monitor="val_AUC",
            mode="max",
            patience=4,
            restore_best_weights=True
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_AUC",
            mode="max",
            factor=0.5,
            patience=2,
            min_lr=1e-6
        )
    ]

history = finetune_model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    callbacks=get_callbacks(),
    batch_size=BATCH_SIZE,
    verbose=1
)

## Plot training and validation history

In [ ]:
plot_training_history(history, "AUC", "loss")

## Evaluate model

In [ ]:
base_model.save("models/finetuned_classification_model.keras")

_, final_val_auc = finetune_model.evaluate(x_val, y_val, verbose=0)
print(f"Validation AUC: {baseline_val_auc:.4f} -> {final_val_auc:.4f}")

loss, auc = base_model.evaluate(x_test, y_test)
print(f"Test loss: {loss:.4f}, Test AUC: {auc:.4f}  (original: 0.9483)")

## Notes